# Saved 3PTB result: receiving public Python player notifications

Owner: [DockingMT #60](https://github.com/uibcdf/dockingmt/issues/60); delivered provider contract: [MolSysViewer #291](https://github.com/uibcdf/molsysviewer/issues/291).

This companion receives exact Viewer source `78ede79e2e345593db5bfe80d246d3f6c70bd005` through its private ordinary-installed wheel. It reuses DockingMT wheel `0.0.0+225.g50d8e33` from producer `50d8e33f4ccb3354a8d27aea9fa9010c1b13e9a9`; no runtime consumer file is changed. The source-free test/data harness is `1c343924b4a144058685796783d72c174b032651`, a distinct qualification identity. Molecular and ArgDigest provider pins remain unchanged. Historical notebooks and receipts are preserved.

Set `DOCKINGMT_NOTEBOOK_ROOT` to that source-free harness, `DOCKINGMT_EXPECTED_PROVIDER_PREFIX` to the independent scientific prefix and `DOCKINGMT_ADDON_OUTPUT` to a new task directory. Jupyter tooling may be task-local; scientific packages must resolve inside the prefix. Headless execution needs a task-owned IPC directory and explicit kernel shutdown; no public port is needed.

The saved five-pose result stays detached. The existing admission recipe explicitly recreates its inputs through MolSysMT: one receptor and two ligand preparation calls. Only the saved rigid/native result is displayed. No new maps, energy, optimization, docking or RMSD is allowed. Original coordinates/identities and writer provenance remain checked separately.

The installation cell binds original artifact identities. Scientific reconstruction and scene cells are identical to the preceding installed companion. The widget cell additionally asserts immediate state after direct Python navigation, before Refresh. A callback confirms accepted Python state, not GPU completion. **Browser controls, real frontend comm delivery, rendered pixels and C6 remain unaccepted.** No current maintainer interaction or public release is required.


In [ ]:
import gzip
import hashlib
import importlib.metadata as metadata
import json
import os
import platform
import subprocess
import sys
from contextlib import ExitStack
from pathlib import Path
from unittest.mock import patch

import dockingmt as dmt
import molsysmt as msm
import molsysviewer as msv
import molsysviewer_dockingmt as addon
import numpy as np
import pyunitwizard as puw
from IPython.display import display
from vina import Vina

root = Path(os.environ['DOCKINGMT_NOTEBOOK_ROOT']).resolve()
prefix = Path(os.environ['DOCKINGMT_EXPECTED_PROVIDER_PREFIX']).resolve()
output = Path(os.environ['DOCKINGMT_ADDON_OUTPUT']).resolve()
expected_producer = '50d8e33f4ccb3354a8d27aea9fa9010c1b13e9a9'
expected_harness = '1c343924b4a144058685796783d72c174b032651'
expected_version = '0.0.0+225.g50d8e33'
expected_wheel_sha256 = '5a923951f5e1bec31f41329b7e62fc9a398286bfba10b6951e984c87b0676c98'
assert not (root / 'dockingmt').exists()
assert not (root / 'molsysviewer_dockingmt').exists()
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=root, text=True).strip() == expected_harness
assert Path(sys.prefix).resolve() == prefix
for module in (dmt, addon, msm, msv):
    assert Path(module.__file__).resolve().is_relative_to(prefix)
distribution = metadata.distribution('dockingmt')
assert Path(distribution.locate_file('')).resolve().is_relative_to(prefix)
assert dmt.__version__ == metadata.version('dockingmt') == expected_version
installed_url = json.loads(distribution.read_text('direct_url.json'))
assert not installed_url.get('dir_info', {}).get('editable', False)
assert installed_url['archive_info']['hashes']['sha256'] == expected_wheel_sha256
entries = [entry for entry in metadata.entry_points(group='molsysviewer.addons')
           if entry.name == 'dockingmt']
assert len(entries) == 1 and entries[0].load() is addon
assert addon.get_addon().version == expected_version
expected_providers = {
    'argdigest': '1cb379478b1abea9d91c06fc6fa5dffa24ede35a24fb1a2d8657c9b13ed55f0c',
    'molsysmt': '4a8e58f40627ff468fab99301674429fba66362f549a31fb4cc747d9cf19a0ba',
    'molsysviewer': '3d31a4d9edb7ed0e72fadcd81b3952196795218f55776478a7625b79e47e9e2a',
}
for name, digest in expected_providers.items():
    provider = metadata.distribution(name)
    assert Path(provider.locate_file('')).resolve().is_relative_to(prefix)
    origin = json.loads(provider.read_text('direct_url.json'))
    assert not origin.get('dir_info', {}).get('editable', False)
    assert origin['archive_info']['hashes']['sha256'] == digest
assert metadata.version('molsysviewer') == '0.24.0+260.g78ede79e'
archive_receipt = json.loads((root / 'devguide/validation/data/license_metadata/checkpoint_2026-11-03.json').read_text())
for name, digest in archive_receipt['canonical_payload_sha256'].items():
    installed = Path(distribution.locate_file(name)).resolve()
    assert installed.is_relative_to(prefix)
    assert hashlib.sha256(installed.read_bytes()).hexdigest() == digest
output.mkdir(parents=True, exist_ok=False)

from devtools.qualify_3ptb_admission import admit
from devtools.recover_saved_3ptb_case import recover_saved_case
from molsysviewer_dockingmt import (
    clear_docking, compute_box_wireframe_coordinate_pairs, ensure_runtime,
    render_search_domain, set_active_pose,
)
from molsysviewer_dockingmt.export import build_docking_export_payload
from molsysviewer_dockingmt.workbench import get_docking_summary_section

def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

frozen_paths = [
    path for family in ('human_3ptb', '3ptb_scene', '3ptb_prospective')
    for path in (root / 'devguide/validation/data' / family).iterdir() if path.is_file()
]
frozen_paths += [root / 'devguide/validation' / name for name in (
    'human_3ptb_workflow_2026-10-20.ipynb', 'human_3ptb_scene_2026-10-23.ipynb')]
frozen = {str(path.relative_to(root)): sha(path) for path in frozen_paths}

forbidden = ExitStack()
blocked_calls = []
def no_computation(*args, **kwargs):
    blocked_calls.append(True)
    raise AssertionError('No new maps, energy, optimization, docking or RMSD in this workflow.')
for name in ('compute_vina_maps', 'score', 'optimize', 'dock'):
    forbidden.enter_context(patch.object(Vina, name, side_effect=no_computation))
for name in ('dock', 'score'):
    forbidden.enter_context(patch.object(dmt, name, side_effect=no_computation))
forbidden.enter_context(patch.object(dmt.DockingResult, 'get_rmsds', side_effect=no_computation))

runtime = {
    'python': platform.python_version(),
    'consumer_source': expected_producer,
    'qualification_harness': expected_harness,
    'provider_archive_sha256': expected_providers,
    'provider_source_received': '78ede79e2e345593db5bfe80d246d3f6c70bd005',
    'consumer_payload_files_verified': 63,
    'consumer_mode': 'installed-wheel',
    'consumer_wheel_sha256': expected_wheel_sha256,
    'editable_installation': False,
    'addon_entry_point_verified': True,
    'consumer_version': dmt.__version__,
    'providers': {name: metadata.version(name) for name in
                  ('molsysmt', 'molsysviewer', 'argdigest', 'pyunitwizard', 'vina')},
    'kernel_tools': {name: metadata.version(name) for name in
                     ('ipykernel', 'jupyter-client', 'jupyter-core', 'pyzmq')},
    'provider_origins_verified': True,
    'consumer_python_files': {
        f'{name}/{path.relative_to(package)}': sha(path)
        for name, package in (('dockingmt', Path(dmt.__file__).parent),
                              ('molsysviewer_dockingmt', Path(addon.__file__).parent))
        for path in sorted(package.rglob('*.py'))
    },
}
print(json.dumps({key: runtime[key] for key in
                  ('python', 'consumer_source', 'consumer_mode', 'providers')}, indent=2))

## Recover the original result, then recreate explicit inputs

`recover_saved_case` authenticates the original bundle and performs no molecular work.
The separate `admit` call repeats the already declared dry-protein/benzamidinium hypothesis.
The comparisons below bind displayed inputs to recorded coordinates, identities, named atom types and charges. New native writer presentation is not substituted into the original backend record.

In [ ]:
bundle = root / 'devguide/validation/data/human_3ptb'
source = root / 'devguide/validation/data/3ptb_prospective/3PTB.pdb.gz'
result, recovery = recover_saved_case(bundle, source)
assert result.problem is None and len(result.poses) == 5
original_result = result.to_dict()
original_admission = json.loads(gzip.decompress((bundle / 'admission_2026-10-20.json.gz').read_bytes()))

with patch.object(dmt, 'prepare_receptor', wraps=dmt.prepare_receptor) as rec_calls, \
     patch.object(dmt, 'prepare_ligand', wraps=dmt.prepare_ligand) as lig_calls:
    receptor, partners, reference, boxes, admission = admit(source)
    preparation_calls = {'receptor': rec_calls.call_count, 'ligand': lig_calls.call_count}
assert preparation_calls == {'receptor': 1, 'ligand': 2}
partner, box = partners['rigid'], boxes['native']
input_checks = {}
for role, prepared in [('receptor', receptor), *partners.items()]:
    old = original_admission['prepared_inputs'][role]
    current = admission['prepared_inputs'][role]
    assert old['coordinate_unit'] == current['coordinate_unit'] == 'angstrom'
    assert old['charge_unit'] == current['charge_unit'] == 'elementary_charge'
    np.testing.assert_allclose(current['coordinates'], old['coordinates'], rtol=0, atol=1e-12)
    for field in ('state_id', 'n_atoms', 'atom_names', 'group_names', 'group_ids', 'atom_types'):
        assert prepared.to_dict()[field] == old['prepared'][field], (role, field)
    np.testing.assert_allclose(prepared.charges, old['prepared']['charges'], rtol=0, atol=1e-12)
    input_checks[role] = {
        'atoms': prepared.n_atoms,
        'coordinates_and_identity_equal': True,
        'original_submitted_sha256': old['pdbqt_sha256'],
        'current_export_sha256': current['pdbqt_sha256'],
        'submitted_bytes_equal': current['pdbqt_sha256'] == old['pdbqt_sha256'],
    }
projection = admission['boxes']['native']['backend_projection']
assert projection == {key: result.provenance['backend_box'][key] for key in projection}
assert result.provenance['backend_box']['rounding_decimals'] == 6
assert np.asarray(admission['protein_full_source_indices']).tolist() == original_admission['protein_full_source_indices']
assert np.asarray(admission['ligand_full_source_indices']).tolist() == original_admission['ligand_full_source_indices']
ref_snapshot = original_admission['ligand']['observed_snapshot']
assert ref_snapshot['structure_units']['coordinates'] == 'nm'
np.testing.assert_allclose(
    puw.get_value(msm.get(reference, coordinates=True), to_unit='nm'),
    ref_snapshot['structures']['coordinates'], rtol=0, atol=1e-12,
)
print(json.dumps({'poses': len(result.poses), 'preparation_calls': preparation_calls,
                  'inputs': input_checks, 'new_searches': 0}, indent=2))

## Build the scene through public providers

Receptor: grey cartoon; selected pose: blue ball-and-stick; observed reference: pink ball-and-stick; search domain: blue wireframe. All five original poses remain separate frames. No alignment or coordinate transformation is applied. The reference has nine observed atoms; it is not expanded into a fictitious thirteen-atom ligand.

In [ ]:
msv.addons.register_module('molsysviewer_dockingmt')
view = dmt.view(result, receptor=receptor, partner=partner,
                reference=reference, search_domain=box)
view.addons.enable('dockingmt')
rec_n, lig_n, ref_n = receptor.n_atoms, partner.n_atoms, msm.get(reference, n_atoms=True)
assert (rec_n, lig_n, ref_n, view.player.n_structures) == (2011, 13, 9, 5)
coordinates = puw.get_value(view.whole.get(coordinates=True), to_unit='angstrom')
assert coordinates.shape == (5, 2033, 3)
rec_coordinates = puw.get_value(receptor.coordinates, to_unit='angstrom')
ref_coordinates = puw.get_value(msm.get(reference, coordinates=True), to_unit='angstrom')[0]
for index, pose in enumerate(result.poses):
    np.testing.assert_allclose(coordinates[index, :rec_n], rec_coordinates, rtol=0, atol=1e-12)
    np.testing.assert_allclose(coordinates[index, rec_n:rec_n+lig_n],
                               puw.get_value(pose.coordinates, to_unit='angstrom'), rtol=0, atol=1e-12)
    np.testing.assert_allclose(coordinates[index, rec_n+lig_n:], ref_coordinates, rtol=0, atol=1e-12)
view.whole.set_representation(preset='empty')
receptor_region = view.regions.add(atom_indices=list(range(rec_n)), tag='receptor',
                                 representation='cartoon', color=0xA8B4C4)
pose_region = view.regions.add(atom_indices=list(range(rec_n, rec_n+lig_n)), tag='selected_pose',
                              representation='ball-and-stick', color=0x2288EE)
ref_region = view.regions.get('reference_ligand')
ref_region.set_representation('ball-and-stick', color=0xDD4499)
for region, system in [
    (receptor_region, receptor.to_molecular_system()),
    (pose_region, result.poses[0].to_molecular_system(partner)),
    (ref_region, reference),
]:
    for field in ('atom_id', 'atom_name', 'atom_type', 'group_id', 'group_name'):
        np.testing.assert_array_equal(region.get(element='atom', **{field: True}),
                                      msm.get(system, element='atom', **{field: True}))
shape = next(item for item in view.shapes.records()
             if item.get('options', {}).get('tag') == 'dockingmt:search_domain')
np.testing.assert_allclose(shape['options']['coordinate_pairs'],
    puw.get_value(compute_box_wireframe_coordinate_pairs(box), to_unit='angstrom'), rtol=0, atol=1e-12)
print({'scene_frames': 5, 'scene_atoms': 2033, 'original_frame_preserved': True})

## Exercise the actual Python widget path

The public resolver creates the same panel classes used by the host. Calling their Python actions and checking the real player's state verifies the kernel boundary. It does not click frontend controls or observe Mol* drawing. Manually resolved probe widgets are closed afterwards; opening the Docking workspace later lets the host create its own window-owned widgets.

In [ ]:
explorer = view.addons.resolve_panel_widget('dockingmt', 'explorer')
domain_panel = view.addons.resolve_panel_widget('dockingmt', 'domain')
try:
    for panel in (explorer, domain_panel):
        panel.on_mount(view)
    for index in (4, 1, 0):
        explorer.handle_action(view, 'set_pose', {'index': index})
        state = explorer.state['explorer']
        assert state['status'] == 'idle' and state['active_index'] == view.player.index == index
        row = state['poses'][index]
        pose = result.poses[index]
        assert row['pose_id'] == pose.pose_id and row['rank'] == pose.rank
        assert {s['name']: s['value'] for s in row['scores']} == pose.scores
        for score in row['scores']:
            definition = pose.score_definitions.get(score['name'])
            assert score['definition'] == definition
            assert score['unit'] == (definition['unit'] if definition else None)
        assert get_docking_summary_section(view)['selected_pose'] == row
        assert build_docking_export_payload(view)['selected_pose'] == row
    explorer.handle_action(view, 'toggle_reference', {})
    assert explorer.state['explorer']['status'] == 'idle' and not ref_region.visible
    explorer.handle_action(view, 'toggle_reference', {})
    assert ref_region.visible
    domain_panel.handle_action(view, 'toggle_domain', {})
    assert domain_panel.state['domain']['status'] == 'idle'
    assert not explorer.state['explorer']['domain_visible']
    domain_panel.handle_action(view, 'toggle_domain', {})
    assert explorer.state['explorer']['domain_visible']
    assert domain_panel.state['domain']['length_unit'] == 'angstrom'
    assert domain_panel.state['domain']['volume_unit'] == 'angstrom**3'
    explorer.on_unmount(view)
    set_active_pose(view, 2)
    explorer.on_mount(view)
    assert explorer.state['explorer']['active_index'] == 1
    navigation_events = []
    view.on_frame_change(navigation_events.append)
    view.player.go_to_structure(4)
    assert explorer.state['explorer']['active_index'] == ensure_runtime(view).active_pose_index == 4
    assert explorer.state['explorer']['poses'][4]['pose_id'] == result.poses[4].pose_id
    assert navigation_events == [{'event': 'frame_changed', 'frame': 4, 'is_playing': False}]
    view.player.go_to_structure(4)
    assert len(navigation_events) == 1
    view.player.go_to_first()
    assert explorer.state['explorer']['active_index'] == 0
    view.off_frame_change(navigation_events.append)
    view.player.go_to_structure(4)
    assert explorer.state['explorer']['active_index'] == 4
    explorer.handle_action(view, 'refresh', {})
    assert explorer.state['explorer']['active_index'] == 4
    set_active_pose(view, 1)
    assert explorer.state['explorer']['active_index'] == 0
    assert 'RMSD unavailable' in explorer.state['explorer']['comparison_note']
finally:
    explorer.close()
    domain_panel.close()

scratch = msv.MolSysView()
try:
    caller_shape = scratch.shapes.add_sphere(center=puw.quantity([0, 0, 0], 'nm'),
                                          radius=puw.quantity(0.1, 'nm'), tag='caller-owned')
    render_search_domain(scratch, box)
    clear_docking(scratch)
    assert scratch.shapes.get('caller-owned') is caller_shape
    assert not scratch.shapes.contains('dockingmt:search_domain')
    assert ensure_runtime(view).result is result and view.player.index == 0
finally:
    scratch.close()
with puw.context(standard_units=['pm', 'fs', 'K', 'mole', 'dalton', 'e', 'kJ/mol', 'degrees']):
    np.testing.assert_allclose(puw.get_value(view.whole.get(coordinates=True), to_unit='angstrom'),
                               coordinates, rtol=0, atol=1e-12)
assert result.to_dict() == original_result
assert frozen == {str(path.relative_to(root)): sha(path) for path in frozen_paths}
assert blocked_calls == []
print({'python_widget_actions_verified': True, 'external_player_refresh_verified': True,
       'external_player_immediate_notifications_verified': True,
       'owned_cleanup_verified': True, 'browser_controls_verified': False})

## Receipt and later visual review

The receipt records this new kernel execution separately from original producers and historical qualification. When a compatible browser session is available, inspect the displayed widget below: select table rows, navigate pose 5 → 1, compare table/scene/score, toggle reference and domain, reopen panels and rotate/zoom the camera. Record actual visibility, clipping and control feedback under #60. These observations are pending; no interactive action from the maintainer is required now.

The notebook leaves the main view available to a live kernel. An automated notebook runner may shut its kernel down after capturing outputs; reopening an executed file alone does not restore a live scientific backend.

In [ ]:
receipt = {
    'schema': 'dockingmt.addon_3ptb_kernel@1',
    'owner': 'uibcdf/dockingmt#60',
    'runtime': runtime,
    'original_producer': recovery['original_producer'],
    'original_result_sha256': recovery['original_result_sha256'],
    'original_qualification_sha256': recovery['qualification_sha256'],
    'source_sha256': sha(source),
    'original_files_verified_unchanged': len(frozen),
    'result_problem_is_none': result.problem is None,
    'pose_population': len(result.poses),
    'molecular_input_replay': {'recipe': 'devtools.qualify_3ptb_admission.admit',
                             'preparation_calls': preparation_calls, 'inputs': input_checks},
    'scene': {'frames': 5, 'atoms': 2033, 'receptor_atoms': rec_n,
              'pose_atoms': lig_n, 'reference_atoms': ref_n, 'selected_index': 0,
              'coordinate_unit': 'angstrom', 'coordinate_tolerance': 1e-12,
              'original_coordinates_and_identities_verified': True,
              'box': admission['boxes']['native']['backend_projection']},
    'new_maps': 0, 'new_searches': 0, 'new_energy_evaluations': 0,
    'new_optimizations': 0, 'new_rmsd_calculations': 0,
    'python_widget_actions_verified': True,
    'public_namespace_and_remount_verified': True,
    'explicit_external_player_refresh_verified': True,
    'immediate_external_player_notifications_verified': True,
    'consecutive_identical_seek_suppressed': True,
    'owned_cleanup_and_view_isolation_verified': True,
    'nondefault_unit_policy_verified': True,
    'browser_controls_verified': False,
    'rendered_pixels_verified': False,
    'maintainer_interactive_review': 'deferred',
    'C6_accepted': False,
    'provider_issue_received': 'uibcdf/molsysviewer#291',
}
with (output / 'receipt.json').open('x') as stream:
    json.dump(receipt, stream, indent=2, allow_nan=False)
forbidden.close()
view.set_panel_mode('addons')
view.set_workspace('dockingmt')
view.set_workspace_panel('explorer', workspace='dockingmt')
display(view)
print(json.dumps({key: receipt[key] for key in (
    'pose_population', 'python_widget_actions_verified', 'new_searches',
    'browser_controls_verified', 'rendered_pixels_verified', 'C6_accepted')}, indent=2))